In [1]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "1e635088-8857-401c-ad7a-bb6863d85fac"
RESOURCE_GROUP = "ML_KNN_PROJECT_BY_UMM-E-KALSOOM"
WORKSPACE = "rg-mlab1-umme-kalsoom"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Connected to: rg-mlab1-umme-kalsoom


In [2]:
# Region mein unique ho, 3-32 characters: letters, numbers, hyphens
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

HttpResponseError: (SubscriptionNotRegistered) Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider
Code: SubscriptionNotRegistered
Message: Resource provider [N/A] isn't registered with Subscription [N/A]. Please see troubleshooting guide, available here: https://aka.ms/register-resource-provider

In [3]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified endpoint [knn-maint-umme-kalsoom] has not been created successfully. Please recreate the endpoint.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-4098e804545d17c2566a581e96fdf4e0-4d45120fdaa9d161-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified endpoint [knn-maint-umme-kalsoom] has not been created successfully. Please recreate the endpoint.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-4098e804545d17c2566a581e96fdf4e0-4d45120fdaa9d161-01\"}"}}
Additional Information:Type: Com

In [4]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

# Pehle dekhein endpoint ki halat kya hai
try:
    ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
    print("State:", ep.provisioning_state)
except Exception as e:
    print("Endpoint nahi mila:", str(e)[:200])

State: EndpointProvisioningState.FAILED


In [5]:
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print("Deleted")

...Deleted


In [6]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance (notebook model)",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created:", ENDPOINT_NAME)

Endpoint created: knn-maint-umme-kalsoom


In [8]:
from azure.ai.ml.entities import ManagedOnlineDeployment

model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

Deploying model: knn-maintenance-notebook version 1


ResourceExistsError: (Conflict) Conflict
Code: Conflict
Message: Conflict
Exception Details:	(InferencingClientCallFailed) "Request could not be completed due to a conflict with the current state of the target resource, Please try again later. Already running method StartCreateDeploymentAsync with operation [13ff90d8-ee16-4703-af20-0df893f29024]. Can not perform StartUpdateDeploymentAsync."
	Code: InferencingClientCallFailed
	Message: "Request could not be completed due to a conflict with the current state of the target resource, Please try again later. Already running method StartCreateDeploymentAsync with operation [13ff90d8-ee16-4703-af20-0df893f29024]. Can not perform StartUpdateDeploymentAsync."

In [ ]:
model = ml_client.models.get(name="knn-maintenance-notebook", label="latest")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,                     # MLflow model: scoring script ki zaroorat nahi
    instance_type="Standard_DS2_v2", # quota error aaye to neeche dekhein
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# 100% requests blue deployment ko bhejein
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment ready")

In [ ]:
print(ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=ENDPOINT_NAME, lines=50))

In [9]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
print("Endpoint state:", ep.provisioning_state)

try:
    d = ml_client.online_deployments.get(name="blue", endpoint_name=ENDPOINT_NAME)
    print("Deployment state:", d.provisioning_state)
except Exception as e:
    print("Deployment abhi nahi mili:", str(e)[:200])

Endpoint state: EndpointProvisioningState.SUCCEEDED
Deployment state: DeploymentProvisioningState.CREATING


In [10]:
ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
ep.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(ep).result()
print("Traffic set: blue = 100%")

HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"DeploymentWeights\":[\"Deployments given positive traffic values should be either in a successful or failed state. Unmatched deployments: [blue]\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-c8de1a5a386b70c44506c8da60c4483b-de85231592086455-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"DeploymentWeights\":[\"Deployments given positive traffic values should be either in a successful or failed state. Unmatched deployments: [blue]\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-c8de1a5a386b70c44506c8da60c4483

In [11]:
import time

ENDPOINT_NAME = "knn-maint-umme-kalsoom"

# blue ki halat dekhte rahein jab tak Succeeded ya Failed na ho
for i in range(60):          # zyada se zyada ~30 minute
    d = ml_client.online_deployments.get(name="blue", endpoint_name=ENDPOINT_NAME)
    state = d.provisioning_state
    print(f"Check {i+1}: deployment state = {state}")
    if state in ("Succeeded", "Failed", "Canceled"):
        break
    time.sleep(30)

# Sirf kamyab hone par traffic dein
if state == "Succeeded":
    ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
    ep.traffic = {"blue": 100}
    ml_client.online_endpoints.begin_create_or_update(ep).result()
    print("Traffic set: blue = 100%")
else:
    print("Deployment kamyab nahi hui, logs dekhein (neeche wala cell)")

Check 1: deployment state = Creating
Check 2: deployment state = Creating
Check 3: deployment state = Creating
Check 4: deployment state = Creating
Check 5: deployment state = Creating
Check 6: deployment state = Creating
Check 7: deployment state = Creating
Check 8: deployment state = Creating
Check 9: deployment state = Creating
Check 10: deployment state = Creating
Check 11: deployment state = Creating
Check 12: deployment state = Creating
Check 13: deployment state = Creating
Check 14: deployment state = Creating
Check 15: deployment state = Creating
Check 16: deployment state = Creating
Check 17: deployment state = Creating
Check 18: deployment state = Creating
Check 19: deployment state = Creating
Check 20: deployment state = Creating
Check 21: deployment state = Creating
Check 22: deployment state = Creating
Check 23: deployment state = Creating
Check 24: deployment state = Creating
Check 25: deployment state = Creating
Check 26: deployment state = Creating
Check 27: deployment 

In [12]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

print(ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=ENDPOINT_NAME, lines=100))

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:04.407907Z, Message: Readiness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:07.288481Z, Message: Readiness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: LivenessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:07.289137Z, Message: Liveness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: LivenessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:19.289083Z, Message: Liveness probe failed: Get \"http://10.66.0.2:5001/\": context deadline exceeded (Client.Timeout exceeded while awaiting headers)
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:19.289171Z, Message: Readiness probe failed: Get \"http://10.66.0.2

In [13]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

print("===== inference-server logs =====")
try:
    print(ml_client.online_deployments.get_logs(
        name="blue", endpoint_name=ENDPOINT_NAME, lines=150,
        container_type="inference-server"))
except Exception as e:
    print("Error:", str(e)[:500])

print("\n===== default logs =====")
try:
    print(ml_client.online_deployments.get_logs(
        name="blue", endpoint_name=ENDPOINT_NAME, lines=150))
except Exception as e:
    print("Error:", str(e)[:500])

===== inference-server logs =====
Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:04.407907Z, Message: Readiness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:07.288481Z, Message: Readiness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: LivenessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:07.289137Z, Message: Liveness probe failed: HTTP probe failed with statuscode: 502
Kind: Pod, Name: LivenessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:19.289083Z, Message: Liveness probe failed: Get \"http://10.66.0.2:5001/\": context deadline exceeded (Client.Timeout exceeded while awaiting headers)
Kind: Pod, Name: ReadinessProbeFailed, Type: Warning, Time: 2026-10-04T07:12:19.289171Z, Message: Readiness pr

In [14]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

ml_client.online_deployments.begin_delete(
    name="blue", endpoint_name=ENDPOINT_NAME).result()
print("blue deleted")

blue deleted


In [15]:
import mlflow, mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "1e635088-8857-401c-ad7a-bb6863d85fac"
RESOURCE_GROUP = "ML_KNN_PROJECT_BY_UMM-E-KALSOOM"
WORKSPACE = "rg-mlab1-umme-kalsoom"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
                     RESOURCE_GROUP, WORKSPACE)
mlflow.set_tracking_uri(ml_client.workspaces.get(WORKSPACE).mlflow_tracking_uri)
mlflow.set_experiment("knn-predictive-maintenance")

# Wohi data aur wohi split (random_state=42), sirf signature ke liye
NUM_COLS = ["air_temp_k", "process_temp_k", "rpm", "torque_nm", "tool_wear_min"]
df = pd.read_csv("../data/ai4i_clean.csv")
df[NUM_COLS] = df[NUM_COLS].astype(float)
X = df[NUM_COLS + ["type"]]
y = df["machine_failure"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# Purana registered model (version 1) load karein
model = mlflow.sklearn.load_model("models:/knn-maintenance-notebook/1")
signature = infer_signature(X_test, model.predict(X_test))

# Naye environment ke saath dobara register (version 2 banega)
with mlflow.start_run(run_name="knn-notebook-deploy-fix") as run:
    mlflow.log_param("note", "re-logged with azureml packages for endpoint")
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=X_test.head(3),
        registered_model_name="knn-maintenance-notebook",
        extra_pip_requirements=[
            "azureml-inference-server-http",
            "azureml-ai-monitoring",
        ],
    )
    print("Run ID:", run.info.run_id)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/04 07:25:34 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - azureml-inference-server-http (current: uninstalled, required: azureml-inference-server-http)
 - azureml-ai-monitoring (current: uninstalled, required:

Run ID: 1aa861d7-1a54-4b0f-a5d0-c31d56d56c3a
🏃 View run knn-notebook-deploy-fix at: https://eastus.api.azureml.ms/mlflow/v2.0/subscriptions/1e635088-8857-401c-ad7a-bb6863d85fac/resourceGroups/ML_KNN_PROJECT_BY_UMM-E-KALSOOM/providers/Microsoft.MachineLearningServices/workspaces/rg-mlab1-umme-kalsoom/#/experiments/403cd18a-2c36-45d8-8c72-34a8997e1dec/runs/1aa861d7-1a54-4b0f-a5d0-c31d56d56c3a
🧪 View experiment at: https://eastus.api.azureml.ms/mlflow/v2.0/subscriptions/1e635088-8857-401c-ad7a-bb6863d85fac/resourceGroups/ML_KNN_PROJECT_BY_UMM-E-KALSOOM/providers/Microsoft.MachineLearningServices/workspaces/rg-mlab1-umme-kalsoom/#/experiments/403cd18a-2c36-45d8-8c72-34a8997e1dec


In [16]:
for m in ml_client.models.list(name="knn-maintenance-notebook"):
    print("Version:", m.version)

Version: 2
Version: 1


In [17]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

ml_client.online_deployments.begin_delete(
    name="blue", endpoint_name=ENDPOINT_NAME).result()
print("blue deleted")

blue deleted


In [18]:
import time
from azure.ai.ml.entities import ManagedOnlineDeployment

ENDPOINT_NAME = "knn-maint-umme-kalsoom"

model = ml_client.models.get(name="knn-maintenance-notebook", version="2")
print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
poller = ml_client.online_deployments.begin_create_or_update(deployment)

for i in range(60):
    d = ml_client.online_deployments.get(name="blue", endpoint_name=ENDPOINT_NAME)
    print(f"Check {i+1}: {d.provisioning_state}")
    if d.provisioning_state in ("Succeeded", "Failed", "Canceled"):
        break
    time.sleep(30)

if d.provisioning_state == "Succeeded":
    ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
    ep.traffic = {"blue": 100}
    ml_client.online_endpoints.begin_create_or_update(ep).result()
    print("Traffic set: blue = 100%  ->  tayyar!")
else:
    print("Phir se fail hui. Logs dekhein.")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint knn-maint-umme-kalsoom exists


Deploying model: knn-maintenance-notebook version 2
.

ResourceNotFoundError: (UserError) Deployment blue not found in endpoint knn-maint-umme-kalsoom, workspace rg-mlab1-umme-kalsoom
Code: UserError
Message: Deployment blue not found in endpoint knn-maint-umme-kalsoom, workspace rg-mlab1-umme-kalsoom

In [19]:
import time
from azure.ai.ml.entities import ManagedOnlineDeployment
from azure.core.exceptions import ResourceNotFoundError

ENDPOINT_NAME = "knn-maint-umme-kalsoom"

def get_state():
    try:
        d = ml_client.online_deployments.get(name="blue", endpoint_name=ENDPOINT_NAME)
        return str(d.provisioning_state)
    except ResourceNotFoundError:
        return None

state = get_state()
print("Abhi blue ki halat:", state)

# blue bilkul nahi hai to ab banayein (is mein 8-15 minute lagte hain)
if state is None:
    model = ml_client.models.get(name="knn-maintenance-notebook", version="2")
    deployment = ManagedOnlineDeployment(
        name="blue",
        endpoint_name=ENDPOINT_NAME,
        model=model,
        instance_type="Standard_DS2_v2",
        instance_count=1,
    )
    ml_client.online_deployments.begin_create_or_update(deployment).result()
    state = get_state()

# Agar pehle se ban rahi hai to intezaar karein
for i in range(60):
    if state and state.lower() in ("succeeded", "failed", "canceled"):
        break
    print(f"Check {i+1}: {state}")
    time.sleep(30)
    state = get_state()

print("Aakhri halat:", state)

if state and state.lower() == "succeeded":
    ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
    ep.traffic = {"blue": 100}
    ml_client.online_endpoints.begin_create_or_update(ep).result()
    print("Traffic set: blue = 100%  ->  tayyar!")

Abhi blue ki halat: DeploymentProvisioningState.CREATING
Check 1: DeploymentProvisioningState.CREATING
......Check 2: DeploymentProvisioningState.CREATING
......Check 3: DeploymentProvisioningState.CREATING
.....Check 4: DeploymentProvisioningState.CREATING
......Check 5: DeploymentProvisioningState.CREATING
......Check 6: DeploymentProvisioningState.CREATING
......Check 7: DeploymentProvisioningState.CREATING
.....Check 8: DeploymentProvisioningState.CREATING
......Check 9: DeploymentProvisioningState.CREATING
......Check 10: DeploymentProvisioningState.CREATING
......Check 11: DeploymentProvisioningState.CREATING
.....Check 12: DeploymentProvisioningState.CREATING
......Check 13: DeploymentProvisioningState.CREATING
......Check 14: DeploymentProvisioningState.CREATING
.....Check 15: DeploymentProvisioningState.CREATING
......Check 16: DeploymentProvisioningState.CREATING
......Check 17: DeploymentProvisioningState.CREATING
......Check 18: DeploymentProvisioningState.CREATING
.....Che

In [20]:
import json, os

sample = {
    "input_data": {
        "columns": ["air_temp_k", "process_temp_k", "rpm", "torque_nm",
                    "tool_wear_min", "type"],
        "index": [0, 1],
        "data": [
            [298.1, 308.6, 1551.0, 42.8, 0.0, "M"],     # sehatmand lagti machine
            [302.5, 310.9, 1325.0, 68.2, 215.0, "L"],   # zyada torque, kam rpm, ghisa tool
        ],
    }
}
os.makedirs("../deployment", exist_ok=True)
with open("../deployment/sample-request.json", "w") as f:
    json.dump(sample, f, indent=2)
print(json.dumps(sample, indent=2))

{
  "input_data": {
    "columns": [
      "air_temp_k",
      "process_temp_k",
      "rpm",
      "torque_nm",
      "tool_wear_min",
      "type"
    ],
    "index": [
      0,
      1
    ],
    "data": [
      [
        298.1,
        308.6,
        1551.0,
        42.8,
        0.0,
        "M"
      ],
      [
        302.5,
        310.9,
        1325.0,
        68.2,
        215.0,
        "L"
      ]
    ]
  }
}


In [21]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="../deployment/sample-request.json",
)
print("Prediction (0 = OK, 1 = failure):", response)

Prediction (0 = OK, 1 = failure): [0, 1]


In [22]:
ENDPOINT_NAME = "knn-maint-umme-kalsoom"

ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
ep.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(ep).result()
print("Traffic set: blue = 100%")